In [1]:
import sys, importlib
sys.path.insert(0, "..")

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("src"):
        del sys.modules[mod_name]

import pandas as pd
import inspect
from src.features import prepare_features
from src.model_loader import load_all_models, predict_ensemble

print("prepare_features signature:")
print(inspect.signature(prepare_features))
print()

models = load_all_models()
print("Models loaded:", list(models.keys()))
print()

raw = pd.read_csv("../data/raw/creditcard.csv")
fraud_row  = raw[raw["Class"] == 1].iloc[[0]].reset_index(drop=True)
normal_row = raw[raw["Class"] == 0].iloc[[0]].reset_index(drop=True)

for name, row in [("FRAUD", fraud_row), ("NORMAL", normal_row)]:
    X_prep = prepare_features(row, models["scaler_amount"], models["scaler_hour"])
    result = predict_ensemble(models, X_prep)

    print(f"=== {name} ===")
    print(f"  ensemble_score : {result['ensemble_score'][0]:.4f}")
    print(f"  xgb_score      : {result['xgb_score'][0]:.4f}")
    print(f"  rf_score       : {result['rf_score'][0]:.4f}")
    print(f"  ae_raw_score   : {result['ae_raw_score'][0]:.4f}")
    print()

prepare_features signature:
(raw_df: pandas.DataFrame, scaler_amount: sklearn.preprocessing._data.RobustScaler, scaler_hour: sklearn.preprocessing._data.RobustScaler) -> pandas.DataFrame



e:\sentinelfraud\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Models loaded: ['xgb', 'rf', 'scaler_amount', 'scaler_hour', 'shap_explainer', 'autoencoder', 'device', 'ensemble_config', 'shap_meta', 'metrics']

=== FRAUD ===
  ensemble_score : 0.9198
  xgb_score      : 0.9998
  rf_score       : 0.6798
  ae_raw_score   : 2.8544

=== NORMAL ===
  ensemble_score : 0.1507
  xgb_score      : 0.0000
  rf_score       : 0.0027
  ae_raw_score   : 0.2165

